# Preprocesamiento — Cancelaciones de reservas hoteleras

- **Proyecto:** Práctica Final · Machine Learning y Deep Learning
- **Máster:** Inteligencia Artificial, Cloud Computing y DevOps · PontIA.tech
- **Fase:** 3 — Preprocesamiento de datos

---

## Objetivo

Implementar y validar el módulo `src/data_loader.py` que prepara los datos para
los modelos. A diferencia del EDA, aquí escribimos código de producción reutilizable
en módulos `.py` y solo usamos este notebook para PROBAR que las funciones funcionan.

## Sub-fases

Este notebook se actualiza incrementalmente a medida que añadimos funcionalidad
al módulo `data_loader.py`:

1. 🔹 Carga del dataset (sub-fase 3.1).
2. 🔹 Limpieza inicial: eliminación de leakage y outliers (sub-fase 3.2).
3. 🔹 Pipeline de transformaciones (sub-fase 3.3).
4. 🔹 Split train/test estratificado (sub-fase 3.4).
5. 🔹 Función orquestadora `preparar_datos()` (sub-fase 3.5).

In [15]:
"""Setup del notebook de preprocesamiento."""

# Configurar el path para poder importar desde src/
import sys
from pathlib import Path

# Subir dos niveles desde notebooks/exploracion/ hasta la raíz del proyecto
PATH_PROYECTO = Path.cwd().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PATH_PROYECTO))

# Imports estándar
import pandas as pd
import numpy as np

# Imports de nuestro módulo
from src.data_loader import cargar_dataset, TARGET_COLUMN, SEED, PATH_DATASET

print(f"Raíz del proyecto: {PATH_PROYECTO}")
print(f"Dataset previsto:  {PATH_DATASET}")
print(f"Target:            {TARGET_COLUMN}")
print(f"SEED:              {SEED}")

Raíz del proyecto: c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml
Dataset previsto:  C:\Juan\Pontia\ML\practica-final-ml\practica-final-ml\data\raw\dataset_practica_final.csv
Target:            is_canceled
SEED:              42


## Sub-fase 3.1 — Carga del dataset

Probamos que la función `cargar_dataset()` del módulo carga correctamente los datos.

In [16]:
"""Prueba de la función cargar_dataset()."""

df = cargar_dataset()

print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Primeras columnas: {list(df.columns[:8])}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

df.head(3)

Dimensiones: 119,390 filas × 32 columnas
Primeras columnas: ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights']
Memoria: 104.83 MB


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


In [17]:
"""Prueba de manejo de error: ruta inválida."""

from pathlib import Path

try:
    cargar_dataset(Path("/ruta/inventada/no_existe.csv"))
except FileNotFoundError as e:
    print(f"✅ Error capturado correctamente: {e}")

✅ Error capturado correctamente: No se encuentra el dataset en \ruta\inventada\no_existe.csv. Asegúrate de que el archivo está en data/raw/.


## Sub-fase 3.2 — Limpieza inicial

Probamos las funciones de limpieza:

- `eliminar_columnas_leakage()`: quita reservation_status, reservation_status_date, assigned_room_type.
- `eliminar_outliers_imposibles()`: quita filas con adr<0 o adults>10.
- `crear_flags_nulos()`: añade tiene_agent y tiene_company.
- `limpiar_dataset()`: orquesta las tres anteriores.

In [18]:
"""Recargar imports del módulo data_loader (por si se modificó)."""

# Recargar el módulo en caso de haber modificado data_loader.py durante esta sesión
import importlib
from src import data_loader
importlib.reload(data_loader)

from src.data_loader import (
    cargar_dataset,
    eliminar_columnas_leakage,
    eliminar_outliers_imposibles,
    crear_flags_nulos,
    limpiar_dataset,
    COLUMNAS_LEAKAGE,
)

print("Funciones importadas correctamente.")
print(f"Columnas con leakage definidas: {COLUMNAS_LEAKAGE}")

Funciones importadas correctamente.
Columnas con leakage definidas: ['reservation_status', 'reservation_status_date', 'assigned_room_type']


In [19]:
"""Probar eliminar_columnas_leakage()."""

df = cargar_dataset()
print(f"Dimensiones antes: {df.shape}")
print(f"¿reservation_status en columnas?: {'reservation_status' in df.columns}")

df_sin_leakage = eliminar_columnas_leakage(df)
print(f"\nDimensiones después: {df_sin_leakage.shape}")
print(f"¿reservation_status en columnas?: {'reservation_status' in df_sin_leakage.columns}")
print(f"¿assigned_room_type en columnas?: {'assigned_room_type' in df_sin_leakage.columns}")

# Comprobar que el DataFrame original NO se modificó
print(f"\n¿El df original sigue intacto?: {'reservation_status' in df.columns}")

Dimensiones antes: (119390, 32)
¿reservation_status en columnas?: True

Dimensiones después: (119390, 29)
¿reservation_status en columnas?: False
¿assigned_room_type en columnas?: False

¿El df original sigue intacto?: True


In [20]:
"""Probar eliminar_outliers_imposibles()."""

print(f"Filas antes: {len(df):,}")
print(f"  - adr < 0:     {(df['adr'] < 0).sum()} filas")
print(f"  - adults > 10: {(df['adults'] > 10).sum()} filas")

df_sin_outliers = eliminar_outliers_imposibles(df)

print(f"\nFilas después: {len(df_sin_outliers):,}")
print(f"  - adr < 0:     {(df_sin_outliers['adr'] < 0).sum()} filas (debe ser 0)")
print(f"  - adults > 10: {(df_sin_outliers['adults'] > 10).sum()} filas (debe ser 0)")

Filas antes: 119,390
  - adr < 0:     1 filas
  - adults > 10: 12 filas
Eliminadas 13 filas con valores imposibles (0.011% del total).

Filas después: 119,377
  - adr < 0:     0 filas (debe ser 0)
  - adults > 10: 0 filas (debe ser 0)


In [21]:
"""Probar crear_flags_nulos()."""

df_con_flags = crear_flags_nulos(df)

print(f"Columnas nuevas creadas: {[c for c in df_con_flags.columns if c not in df.columns]}")
print(f"\nDistribución de tiene_agent:")
print(df_con_flags["tiene_agent"].value_counts())
print(f"\nDistribución de tiene_company:")
print(df_con_flags["tiene_company"].value_counts())

# Confirmar correspondencia con los nulos
print(f"\nVerificación:")
print(f"  tiene_agent=0 ({(df_con_flags['tiene_agent']==0).sum():,}) vs agent nulos ({df['agent'].isnull().sum():,})")
print(f"  tiene_company=0 ({(df_con_flags['tiene_company']==0).sum():,}) vs company nulos ({df['company'].isnull().sum():,})")

Columnas nuevas creadas: ['tiene_agent', 'tiene_company']

Distribución de tiene_agent:
tiene_agent
1    103050
0     16340
Name: count, dtype: int64

Distribución de tiene_company:
tiene_company
0    112593
1      6797
Name: count, dtype: int64

Verificación:
  tiene_agent=0 (16,340) vs agent nulos (16,340)
  tiene_company=0 (112,593) vs company nulos (112,593)


In [22]:
"""Probar limpiar_dataset() (la función orquestadora)."""

df_original = cargar_dataset()
df_limpio = limpiar_dataset(df_original)

print(f"Dimensiones originales: {df_original.shape}")
print(f"Dimensiones tras limpieza: {df_limpio.shape}")
print(f"\nColumnas eliminadas: {set(df_original.columns) - set(df_limpio.columns)}")
print(f"Columnas añadidas:   {set(df_limpio.columns) - set(df_original.columns)}")

print(f"\nResumen de la limpieza:")
print(f"  - Filas: {df_original.shape[0]:,} → {df_limpio.shape[0]:,}")
print(f"  - Columnas: {df_original.shape[1]} → {df_limpio.shape[1]}")

Eliminadas 13 filas con valores imposibles (0.011% del total).
Dimensiones originales: (119390, 32)
Dimensiones tras limpieza: (119377, 31)

Columnas eliminadas: {'reservation_status', 'assigned_room_type', 'reservation_status_date'}
Columnas añadidas:   {'tiene_agent', 'tiene_company'}

Resumen de la limpieza:
  - Filas: 119,390 → 119,377
  - Columnas: 32 → 31


**Resultados de la sub-fase 3.2:**

- Las 3 funciones de limpieza funcionan correctamente.
- La función orquestadora `limpiar_dataset()` aplica todas en orden.
- Las funciones devuelven copias, no modifican el DataFrame original.
- El dataset limpio pasa de 32 columnas a ~31 (eliminamos 3 con leakage, añadimos 2 flags).
- Se eliminan unas pocas filas con valores imposibles (~3-5 filas).

**Listo para la siguiente sub-fase: pipeline de transformaciones (sub-fase 3.3).**

In [23]:
"""Verificación rápida del estado actual del módulo."""

import importlib
from src import data_loader
importlib.reload(data_loader)

# Comprobar qué constantes y funciones están definidas
constantes_esperadas = [
    "NUMERICAS_REALES",
    "NUMERICAS_LOG",
    "NUMERICAS_PASS_THROUGH",
    "CATEGORICAS_BAJA_CARDINALIDAD",
    "CATEGORICAS_ALTA_CARDINALIDAD",
    "COLUMNAS_TEMPORALES",
]

for c in constantes_esperadas:
    existe = hasattr(data_loader, c)
    print(f"  {c}: {'✅ existe' if existe else '❌ falta'}")

  NUMERICAS_REALES: ✅ existe
  NUMERICAS_LOG: ✅ existe
  NUMERICAS_PASS_THROUGH: ✅ existe
  CATEGORICAS_BAJA_CARDINALIDAD: ✅ existe
  CATEGORICAS_ALTA_CARDINALIDAD: ✅ existe
  COLUMNAS_TEMPORALES: ✅ existe


## Sub-fase 3.3 — Pipeline de transformaciones con ColumnTransformer

Hasta ahora todo el preprocesamiento se ha hecho con funciones simples
(eliminar columnas, filtrar filas, crear flags). Ahora damos el salto a
**transformaciones encapsuladas en un objeto sklearn** que aprende parámetros
del train y los aplica al test sin verlo. Esto evita data leakage automáticamente.

**Lo que vamos a construir**:

- Listas de constantes que clasifican cada columna por su tipo (numérica con log,
  numérica normal, categórica baja cardinalidad, alta cardinalidad, etc.).
- Una función `aplicar_top_n_other()` que reduce la cardinalidad de country y
  agent (de 178 y 334 a 31 y 31 respectivamente).
- Un `ColumnTransformer` con 6 ramas, una por cada tipo de columna.

**Salida esperada**: un objeto preprocesador listo para `fit_transform()` que
convierte las 30 columnas originales en aproximadamente 130 features numéricas
y escaladas.

In [35]:
"""Probar el ColumnTransformer."""

import importlib
from src import data_loader
importlib.reload(data_loader)

from src.data_loader import (
    cargar_dataset,
    limpiar_dataset,
    aplicar_top_n_other,
    construir_preprocesador,
    TARGET_COLUMN,
)

# 1. Cargar y limpiar
df = cargar_dataset()
df_limpio = limpiar_dataset(df)
print(f"Tras limpieza: {df_limpio.shape}")

# 2. Aplicar reducción top-N + Other
df_reducido = aplicar_top_n_other(df_limpio)
print(f"Tras top-N + Other: {df_reducido.shape}")
print(f"  country únicos:  {df_reducido['country'].nunique()} (debería ser ~31: top-30 + Other)")
print(f"  agent únicos:    {df_reducido['agent'].nunique()} (debería ser ~31)")

# 3. Separar X e y
X = df_reducido.drop(columns=[TARGET_COLUMN])
y = df_reducido[TARGET_COLUMN]
print(f"\nX shape: {X.shape}, y shape: {y.shape}")

# 4. Construir preprocesador y aplicarlo
preprocesador = construir_preprocesador()
X_transformado = preprocesador.fit_transform(X)

print(f"\nX tras preprocesador: {X_transformado.shape}")
print(f"  - Filas: {X_transformado.shape[0]:,}")
print(f"  - Columnas: {X_transformado.shape[1]} (era 31, se expanden por OneHot)")
print(f"\nTipo del output: {type(X_transformado).__name__}")
print(f"¿Tiene nulos?: {np.isnan(X_transformado).any() if hasattr(X_transformado, 'shape') else 'N/A'}")

Eliminadas 13 filas con valores imposibles (0.011% del total).
Tras limpieza: (119377, 31)
Tras top-N + Other: (119377, 31)
  country únicos:  31 (debería ser ~31: top-30 + Other)
  agent únicos:    31 (debería ser ~31)

X shape: (119377, 30), y shape: (119377,)

X tras preprocesador: (119377, 130)
  - Filas: 119,377
  - Columnas: 130 (era 31, se expanden por OneHot)

Tipo del output: ndarray
¿Tiene nulos?: False


### Aplicación del preprocesador y verificación del output

Construimos el `ColumnTransformer` y lo aplicamos sobre el dataset completo
(aún no hacemos split). Esto nos permite verificar que el pipeline funciona
antes de meterlo en producción.

**Importante**: en esta celda usamos `fit_transform(X)` sobre el conjunto
completo SOLO para validar dimensiones. En el flujo real (sub-fase 3.5),
haremos primero el split y luego `fit_transform(X_train)` + `transform(X_test)`.

### Verificación de calidad del output

Comprobamos cinco propiedades del output del preprocesador:

1. **Dimensiones**: las filas se conservan, las columnas se expanden por OneHot.
2. **Nulos**: 0 nulos en el output (todo imputado).
3. **Escalado**: las numéricas tienen media ~0 y std ~1.
4. **One-hot**: las columnas categóricas son binarias 0/1.
5. **Pass-through**: las binarias (tiene_agent, tiene_company, is_repeated_guest)
   pasan sin tocar.

Si las cinco verificaciones salen OK, el preprocesador está listo para usarse
en datos nuevos sin sorpresas.

In [26]:
"""Verificación exhaustiva del output del preprocesador."""

import numpy as np

# Recargar e importar las constantes necesarias
import importlib
from src import data_loader
importlib.reload(data_loader)

from src.data_loader import (
    NUMERICAS_REALES,
    COLUMNAS_TEMPORALES,
    NUMERICAS_PASS_THROUGH,
)

# Convertir a DataFrame para inspeccionar (con nombres de columna)
feature_names = preprocesador.get_feature_names_out()
X_df = pd.DataFrame(X_transformado, columns=feature_names)

print("=" * 70)
print("VERIFICACION 1 - Dimensiones")
print("=" * 70)
print(f"Input:  {X.shape[0]:,} filas x {X.shape[1]} columnas")
print(f"Output: {X_df.shape[0]:,} filas x {X_df.shape[1]} columnas")
print(f"Filas conservadas: {'OK' if X.shape[0] == X_df.shape[0] else 'FAIL'}")

print("\n" + "=" * 70)
print("VERIFICACION 2 - Nulos y valores anomalos")
print("=" * 70)
n_nulos = X_df.isnull().sum().sum()
n_inf = np.isinf(X_df.values).sum()
print(f"Total nulos en output: {n_nulos}")
print(f"Total infinitos en output: {n_inf}")
print(f"Sin nulos:     {'OK' if n_nulos == 0 else 'FAIL'}")
print(f"Sin infinitos: {'OK' if n_inf == 0 else 'FAIL'}")

print("\n" + "=" * 70)
print("VERIFICACION 3 - Escalado de numericas")
print("=" * 70)
columnas_numericas_escaladas = [c for c in feature_names
                                  if c in (NUMERICAS_REALES + COLUMNAS_TEMPORALES)]

medias = X_df[columnas_numericas_escaladas].mean()
stds = X_df[columnas_numericas_escaladas].std()

print(f"Columnas numericas escaladas: {len(columnas_numericas_escaladas)}")
print(f"\nMedias (esperadas ~0):")
print(medias.describe()[["min", "max", "mean"]].round(4))
print(f"\nDesviaciones tipicas (esperadas ~1):")
print(stds.describe()[["min", "max", "mean"]].round(4))

medias_ok = (medias.abs() < 0.01).all()
stds_ok = (stds.between(0.99, 1.01)).all()
print(f"\nMedias cercanas a 0: {'OK' if medias_ok else 'WARN - ver detalle'}")
print(f"Std cercanas a 1:    {'OK' if stds_ok else 'WARN - ver detalle'}")

print("\n" + "=" * 70)
print("VERIFICACION 4 - One-hot encoding")
print("=" * 70)
columnas_one_hot = [c for c in feature_names
                     if c not in (NUMERICAS_REALES + COLUMNAS_TEMPORALES + NUMERICAS_PASS_THROUGH)]
print(f"Columnas creadas por OneHot: {len(columnas_one_hot)}")

valores_unicos = set()
for col in columnas_one_hot[:5]:
    valores_unicos.update(X_df[col].unique())

print(f"Valores unicos (muestreo 5 cols): {sorted(valores_unicos)}")
print(f"Son solo 0 y 1: {'OK' if valores_unicos.issubset({0, 1, 0.0, 1.0}) else 'FAIL'}")

print("\n" + "=" * 70)
print("VERIFICACION 5 - Pass-through (binarias sin tocar)")
print("=" * 70)
for col in NUMERICAS_PASS_THROUGH:
    if col in feature_names:
        valores = sorted(X_df[col].unique())
        print(f"  {col}: valores unicos = {valores}")

print("\n" + "=" * 70)
print("RESUMEN")
print("=" * 70)
print(f"Preprocesador funcionando correctamente.")
print(f"{X_df.shape[1]} features listas para los modelos.")
print(f"{X_df.shape[0]:,} muestras sin perdida de filas.")

VERIFICACION 1 - Dimensiones
Input:  119,377 filas x 30 columnas
Output: 119,377 filas x 130 columnas
Filas conservadas: OK

VERIFICACION 2 - Nulos y valores anomalos
Total nulos en output: 0
Total infinitos en output: 0
Sin nulos:     OK
Sin infinitos: OK

VERIFICACION 3 - Escalado de numericas
Columnas numericas escaladas: 16

Medias (esperadas ~0):
min    -0.0
max     0.0
mean    0.0
dtype: float64

Desviaciones tipicas (esperadas ~1):
min     1.0
max     1.0
mean    1.0
dtype: float64

Medias cercanas a 0: OK
Std cercanas a 1:    OK

VERIFICACION 4 - One-hot encoding
Columnas creadas por OneHot: 111
Valores unicos (muestreo 5 cols): [0.0, 1.0]
Son solo 0 y 1: OK

VERIFICACION 5 - Pass-through (binarias sin tocar)
  tiene_agent: valores unicos = [0.0, 1.0]
  tiene_company: valores unicos = [0.0, 1.0]
  is_repeated_guest: valores unicos = [0.0, 1.0]

RESUMEN
Preprocesador funcionando correctamente.
130 features listas para los modelos.
119,377 muestras sin perdida de filas.


In [27]:
"""Probar la funcion orquestadora del preprocesamiento."""

import importlib
from src import data_loader
importlib.reload(data_loader)

from src.data_loader import aplicar_preprocesamiento_completo, construir_preprocesador

# Una sola llamada hace todo el preprocesamiento previo al split
X, y = aplicar_preprocesamiento_completo()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nDistribucion del target:")
print(y.value_counts(normalize=True).round(3))

# El ColumnTransformer aun se aplica aparte (lo haremos despues del split)
preprocesador = construir_preprocesador()
print(f"\nPreprocesador construido: {type(preprocesador).__name__}")
print(f"  - Transformers: {len(preprocesador.transformers)}")

Eliminadas 13 filas con valores imposibles (0.011% del total).
X shape: (119377, 30)
y shape: (119377,)

Distribucion del target:
is_canceled
0    0.63
1    0.37
Name: proportion, dtype: float64

Preprocesador construido: ColumnTransformer
  - Transformers: 6


## Sub-fase 3.4 — Split train/test estratificado

Dividimos los datos en dos conjuntos: 80% train (entrenamiento) y 20% test
(evaluación final). Con dos peculiaridades importantes:

- **Estratificado** (`stratify=y`): se garantiza que la proporción 63/37 del
  target se mantenga exactamente igual en train y test. Sin estratificar, el
  split aleatorio podría dejar 70/30 en train y 55/45 en test por azar, lo
  cual distorsionaría las métricas.
- **Reproducible** (`random_state=42`): la semilla fija permite que cada
  ejecución del notebook produzca el mismo split. Crítico para que los
  resultados sean comparables y replicables.

**Importante**: aún NO aplicamos el `ColumnTransformer`. Esto se hace DESPUÉS
del split, dentro de la función orquestadora final. Si lo hiciéramos ahora,
el escalado vería los datos de test y filtraría información (data leakage
clásico de train-test).

In [28]:
"""Probar split_train_test_estratificado()."""

import importlib
from src import data_loader
importlib.reload(data_loader)

from src.data_loader import (
    aplicar_preprocesamiento_completo,
    split_train_test_estratificado,
)

# 1. Obtener X e y
X, y = aplicar_preprocesamiento_completo()

# 2. Hacer el split
X_train, X_test, y_train, y_test = split_train_test_estratificado(X, y)

# 3. Verificar
print("=" * 70)
print("VERIFICACION DEL SPLIT")
print("=" * 70)

print(f"\nDimensiones:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

print(f"\nProporciones:")
total = len(X_train) + len(X_test)
print(f"  Train: {len(X_train) / total * 100:.1f}% ({len(X_train):,} filas)")
print(f"  Test:  {len(X_test) / total * 100:.1f}% ({len(X_test):,} filas)")

print(f"\nDistribucion del target:")
print(f"  Global: {y.value_counts(normalize=True).round(4).to_dict()}")
print(f"  Train:  {y_train.value_counts(normalize=True).round(4).to_dict()}")
print(f"  Test:   {y_test.value_counts(normalize=True).round(4).to_dict()}")

# Verificar que la estratificacion funciono
diff_train = abs(y_train.mean() - y.mean())
diff_test = abs(y_test.mean() - y.mean())
print(f"\nDiferencias respecto al global:")
print(f"  Train: {diff_train:.4f} (deberia ser ~0)")
print(f"  Test:  {diff_test:.4f} (deberia ser ~0)")

estratif_ok = diff_train < 0.001 and diff_test < 0.001
print(f"\nEstratificacion correcta: {'OK' if estratif_ok else 'FAIL'}")

Eliminadas 13 filas con valores imposibles (0.011% del total).
VERIFICACION DEL SPLIT

Dimensiones:
  X_train: (95501, 30)
  X_test:  (23876, 30)
  y_train: (95501,)
  y_test:  (23876,)

Proporciones:
  Train: 80.0% (95,501 filas)
  Test:  20.0% (23,876 filas)

Distribucion del target:
  Global: {0: 0.6296, 1: 0.3704}
  Train:  {0: 0.6296, 1: 0.3704}
  Test:   {0: 0.6296, 1: 0.3704}

Diferencias respecto al global:
  Train: 0.0000 (deberia ser ~0)
  Test:  0.0000 (deberia ser ~0)

Estratificacion correcta: OK


## Sub-fase 3.5 — Función orquestadora final `preparar_datos()`

Esta es la culminación de toda la Fase 3. Una sola función que encapsula
TODO el pipeline de preprocesamiento desde el CSV crudo hasta X_train,
X_test, y_train, y_test listos para entrenar modelos.

**Orden interno de operaciones**:

1. Cargar el dataset.
2. Limpiar (eliminar leakage, outliers, crear flags).
3. Aplicar top-N + Other a country y agent.
4. Separar X e y.
5. Split train/test estratificado.
6. Construir el ColumnTransformer.
7. `fit_transform()` en X_train (aprende parámetros del train).
8. `transform()` en X_test (aplica lo aprendido sin verlo).

**Por qué este orden importa**: pasos 1-4 no usan estadísticas. El split
(paso 5) separa los datos. El `fit_transform` (paso 7) aprende SOLO de train.
El `transform` (paso 8) aplica lo aprendido al test. Así garantizamos cero
data leakage del test al train.

**Salida**: `(X_train, X_test, y_train, y_test, preprocesador)` donde el
preprocesador es un objeto ya entrenado que podemos reusar en producción
con datos nuevos.

**Resultado final**: Fase 3 completada. Listos para Fase 4 (Modelado).

In [32]:
"""Probar preparar_datos() - la funcion orquestadora final."""

import importlib
from src import data_loader
importlib.reload(data_loader)

from src.data_loader import preparar_datos

# Una sola llamada hace TODO el preprocesamiento
X_train, X_test, y_train, y_test, preprocesador = preparar_datos()

print("=" * 70)
print("PIPELINE COMPLETO EJECUTADO")
print("=" * 70)

print(f"\nDimensiones finales:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

print(f"\nTipo del output:")
print(f"  X_train: {type(X_train).__name__}")
print(f"  X_test:  {type(X_test).__name__}")

print(f"\nProporcion del target:")
print(f"  Train: cancela={y_train.mean()*100:.2f}%")
print(f"  Test:  cancela={y_test.mean()*100:.2f}%")

print(f"\nPreprocesador devuelto:")
print(f"  Tipo: {type(preprocesador).__name__}")
print(f"  Esta entrenado: {hasattr(preprocesador, 'transformers_')}")

# Verificar que no hay nulos ni infinitos en train ni test
import numpy as np
n_nulos_train = np.isnan(X_train).sum()
n_nulos_test = np.isnan(X_test).sum()
print(f"\nValidacion final:")
print(f"  Nulos en X_train: {n_nulos_train} ({'OK' if n_nulos_train == 0 else 'FAIL'})")
print(f"  Nulos en X_test:  {n_nulos_test} ({'OK' if n_nulos_test == 0 else 'FAIL'})")

print(f"\nLISTO PARA FASE 4 (MODELADO)")

Eliminadas 13 filas con valores imposibles (0.011% del total).
PIPELINE COMPLETO EJECUTADO

Dimensiones finales:
  X_train: (95501, 130)
  X_test:  (23876, 130)
  y_train: (95501,)
  y_test:  (23876,)

Tipo del output:
  X_train: ndarray
  X_test:  ndarray

Proporcion del target:
  Train: cancela=37.04%
  Test:  cancela=37.04%

Preprocesador devuelto:
  Tipo: ColumnTransformer
  Esta entrenado: True

Validacion final:
  Nulos en X_train: 0 (OK)
  Nulos en X_test:  0 (OK)

LISTO PARA FASE 4 (MODELADO)


In [33]:
"""Probar preparar_datos() devolviendo DataFrames con nombres de columna."""

X_train_df, X_test_df, y_train, y_test, preprocesador = preparar_datos(devolver_dataframes=True)

print(f"X_train tipo: {type(X_train_df).__name__}")
print(f"X_train shape: {X_train_df.shape}")
print(f"\nPrimeras 5 columnas:")
print(X_train_df.columns[:5].tolist())
print(f"\nUltimas 5 columnas:")
print(X_train_df.columns[-5:].tolist())
print(f"\nMuestra de 3 filas:")
X_train_df.head(3)

Eliminadas 13 filas con valores imposibles (0.011% del total).
X_train tipo: DataFrame
X_train shape: (95501, 130)

Primeras 5 columnas:
['lead_time', 'adr', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults']

Ultimas 5 columnas:
['arrival_date_week_number', 'arrival_date_day_of_month', 'tiene_agent', 'tiene_company', 'is_repeated_guest']

Muestra de 3 filas:


,lead_time,adr,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,previous_cancellations,previous_bookings_not_canceled,booking_changes,...,agent_9.0,agent_96.0,agent_Other,agent_sin_agent,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,tiene_agent,tiene_company,is_repeated_guest
107086,0.675980,-0.110492,0.074872,-0.786051,0.301529,-0.260395,-0.080827,-0.103579,-0.090657,-0.339020,...,0.0,0.0,0.0,0.0,1.192937,-1.258000,-1.113982,1.0,0.0,0.0
33728,-1.173852,-0.706421,1.076879,-0.261040,0.301529,-0.260395,-0.080827,-0.103579,-0.090657,2.716973,...,0.0,0.0,0.0,0.0,1.192937,-1.331476,1.161675,1.0,0.0,0.0
119330,0.833908,0.602870,-0.927135,1.313992,0.301529,-0.260395,-0.080827,-0.103579,-0.090657,-0.339020,...,1.0,0.0,0.0,0.0,1.192937,0.578904,1.503024,1.0,0.0,0.0
